# RQ2 — Implementation Practices (Q17–Q20)

**Analysis to Address RQ2**: Q17–Q20 — how data quality is incorporated into development, how its impact is measured, how often requirements are formally discussed, and how issues are documented and communicated.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import utils as U

U.setup_matplotlib()
TABLES = U.DATA_PROC / "tables"
TABLES.mkdir(exist_ok=True)

df = U.load_anonymized()
print(f"N={len(df)}")


## Canonical options and helper functions

In [ ]:
Q17_OPTIONS = {
    "q17_initial_eval": [
        "Avaliação inicial durante a coleta e preparação de dados",
        "Initial assessment during data collection and preparation",
    ],
    "q17_continuous_mon": [
        "Monitoramento contínuo durante todo o ciclo de vida do modelo",
        "Continuous monitoring throughout the model's life cycle",
    ],
    "q17_test_sets": [
        "Conjuntos de testes são aplicados para validar a consistência, completude e precisão dos dados antes de serem usados no treinamento.",
        "Test sets are applied to validate the consistency, completeness and accuracy of the data before it is used for training.",
    ],
    "q17_no_strategy": [
        "Não existe uma estratégia formal para assegurar a qualidade dos dados durante o desenvolvimento.",
        "There is no formal strategy to ensure data quality during development.",
    ],
}
Q17_LABELS = {
    "q17_initial_eval": "Initial assessment in collection/preparation",
    "q17_continuous_mon": "Continuous monitoring across lifecycle",
    "q17_test_sets": "Test sets before training",
    "q17_no_strategy": "No formal strategy",
}

Q18_OPTIONS = {
    "q18_ab_tests": ["Testes A/B", "A/B testing"],
    "q18_perf_metrics": [
        "Análise de métricas de performance (ex.: precisão, recall)",
        "Analysis of performance metrics (e.g. precision, recall)",
    ],
    "q18_manual_review": ["Revisão manual dos resultados", "Manual review of results"],
}
Q18_LABELS = {
    "q18_ab_tests": "A/B testing",
    "q18_perf_metrics": "Performance metrics",
    "q18_manual_review": "Manual review of results",
}

Q20_OPTIONS = {
    "q20_structured_text": ["Linguagem estruturada (texto)", "Structured language (text)"],
    "q20_pm_tools": [
        "Ferramentas de Gerenciamento de Projetos (Jira, Trello ou Asana)",
        "Project management tools (Jira, Trello or Asana)",
    ],
    "q20_central_docs": [
        "Documentação Centralizada (Sistemas como Confluence, Google Docs ou Notion)",
        "Centralized documentation (systems such as Confluence, Google Docs or Notion)",
    ],
    "q20_alignment_meet": ["Reuniões de Alinhamento", "Alignment meetings"],
    "q20_periodic_reports": ["Relatórios Periódicos", "Periodic reports"],
}
Q20_LABELS = {
    "q20_structured_text": "Structured language (text)",
    "q20_pm_tools": "Project management tools (Jira/etc.)",
    "q20_central_docs": "Centralized documentation (Confluence/etc.)",
    "q20_alignment_meet": "Alignment meetings",
    "q20_periodic_reports": "Periodic reports",
}


## 2. Parser — converts free text into binary columns

In [ ]:
rng = np.random.default_rng(42)

def _boot_ci(
    values: np.ndarray,
    threshold: int = 4,
    n_bootstrap: int = 2000,
    ci_level: float = 0.95,
) -> tuple[float, float]:
    """Percentile bootstrap CI for the proportion of values >= threshold.

    Parameters
    ----------
    values      : 1-D array of observed values (int or bool/0-1)
    threshold   : values >= threshold count as "success" (default 4, Likert top-box)
                  For binary 0/1 arrays pass threshold=1.
    n_bootstrap : number of bootstrap resamples (default 2000)
    ci_level    : confidence level (default 0.95 → 95% CI)

    Returns
    -------
    (lo, hi) proportions in [0, 1]
    """
    values = np.asarray(values)
    n = len(values)
    if n == 0:
        return float("nan"), float("nan")

    alpha = 1 - ci_level
    boot_indices = rng.integers(0, n, size=(n_bootstrap, n))
    boot_props = np.mean(values[boot_indices] >= threshold, axis=1)
    lo = np.percentile(boot_props, 100 * alpha / 2)
    hi = np.percentile(boot_props, 100 * (1 - alpha / 2))
    return lo, hi

In [ ]:
def parse_checkboxes(series: pd.Series, options: dict) -> tuple[pd.DataFrame, pd.Series]:
    """For each response, marks True if *any* alias (PT/EN) of the option appears
    as a substring. `options` accepts `{key: str}` or `{key: list[str]}`.
    Returns (binary df, series with residual text after removing the options)."""
    binary = pd.DataFrame(index=series.index, columns=list(options.keys()), dtype=bool)
    binary[:] = False
    residual = series.copy()
    for key, raw in options.items():
        labels = [raw] if isinstance(raw, str) else list(raw)
        present = pd.Series(False, index=series.index)
        for lab in labels:
            present = present | series.fillna("").str.contains(lab, regex=False)
            residual = residual.fillna("").str.replace(lab, "", regex=False)
        binary[key] = present
    residual = residual.str.replace(r"^[,\.\s]+|[,\.\s]+$", "", regex=True)
    residual = residual.str.replace(r"^[,\.]\s*", "", regex=True)
    residual = residual.where(residual.str.len() > 2, "")
    return binary, residual


def proportions_with_ci(binary: pd.DataFrame, labels: dict[str, str], n_total: int) -> pd.DataFrame:
    rows = []
    for key in binary.columns:
        vals = binary[key].astype(int).values
        lo, hi = _boot_ci(vals)
        rows.append({
            "key": key,
            "label": labels[key],
            "n": int(binary[key].sum()),
            "pct": binary[key].mean() * 100,
            "ci_lo": lo * 100,
            "ci_hi": hi * 100,
        })
    return pd.DataFrame(rows).sort_values("pct", ascending=False).reset_index(drop=True)

## Parsing checkbox responses

In [ ]:
q17_bin, q17_res = parse_checkboxes(df["incorporation_open"], Q17_OPTIONS)
q18_bin, q18_res = parse_checkboxes(df["measurement_open"], Q18_OPTIONS)
q20_bin, q20_res = parse_checkboxes(df["documentation_open"], Q20_OPTIONS)

n_q17 = df["incorporation_open"].notna().sum()
n_q18 = df["measurement_open"].notna().sum()
n_q20 = df["documentation_open"].notna().sum()

for q, name, n in [(q17_bin, "Q17", n_q17), (q18_bin, "Q18", n_q18), (q20_bin, "Q20", n_q20)]:
    print(f"{name}: n={n}, average options marked = {q.sum(axis=1).mean():.2f}")


In [ ]:
# Text residual (P13 Slack/P18 "in the same MR"/etc.)
for q, name in [(q17_res, "Q17"), (q18_res, "Q18"), (q20_res, "Q20"), (q21_res, "Q21")]:
    nonempty = q[q.str.len() > 0]
    if len(nonempty):
        print(f"=== {name} residuals ===")
        for idx, txt in nonempty.items():
            print(f"  P{idx:02d}: {txt!r}")

## 3. Proportion tables with 95% CI

In [ ]:
p17 = proportions_with_ci(q17_bin, Q17_LABELS, n_q17)
p18 = proportions_with_ci(q18_bin, Q18_LABELS, n_q18)
p20 = proportions_with_ci(q20_bin, Q20_LABELS, n_q20)


## Q17 + Q18 — Incorporation and Measurement

In [ ]:
def horizontal_bar(p: pd.DataFrame, title: str, color: str, ax) -> None:
    p_sorted = p.sort_values("pct")
    y = np.arange(len(p_sorted))
    ax.barh(y, p_sorted["pct"], color=color, edgecolor="white")

    err_lo = np.clip(p_sorted["pct"] - p_sorted["ci_lo"], 0, None)
    err_hi = np.clip(p_sorted["ci_hi"] - p_sorted["pct"], 0, None)

    ax.errorbar(p_sorted["pct"], y,
                xerr=[err_lo, err_hi],
                fmt="none", ecolor="black", elinewidth=0.6, capsize=2)
    ax.set_yticks(y)
    ax.set_yticklabels(p_sorted["label"])
    ax.set_xlabel("% of responses")
    ax.set_xlim(0, 100)
    ax.set_title(title)
    for i, (pct, n) in enumerate(zip(p_sorted["pct"], p_sorted["n"])):
        ax.text(pct + 1, i, f"{int(n)}", va="center", fontsize=7)

fig, axes = plt.subplots(2, 1, figsize=(7.0, 5.5))
horizontal_bar(p17, f"Q17 — How data quality is incorporated (n={n_q17})", U.PALETTE_WONG[2], axes[0])
horizontal_bar(p18, f"Q18 — How model-quality impact is measured (n={n_q18})", U.PALETTE_WONG[3], axes[1])
fig.tight_layout()
U.save_fig(fig, "implementation_q17_q18")
plt.show()

## Q20 — Documentation and Communication

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7.0, 3.5))
horizontal_bar(p20, f"Q20 — How data quality is documented/communicated (n={n_q20})", U.PALETTE_WONG[1], ax)
fig.tight_layout()
U.save_fig(fig, "documentation_q20")
plt.show()


## Q19 — Frequency of Formal Discussion

In [ ]:
disc_labels = ["Never", "<1x/month", "1x/month to <1x/week", ">=1x/week", "Daily"]

fig, ax = plt.subplots(figsize=(6.0, 2.8))
col, labels, title = (
    "discussion_freq", disc_labels,
    "Q19 — Formal discussion frequency of quality requirements"
)
counts = df[col].value_counts(dropna=False).reindex(range(1, len(labels) + 1), fill_value=0)
pct = counts / counts.sum() * 100
ax.bar(range(len(labels)), pct.values, color=U.PALETTE_WONG[2], edgecolor="white")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("% of responses")
ax.set_title(title)
ax.set_ylim(0, max(pct.values) * 1.15)
for i, v in enumerate(pct.values):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center", va="bottom", fontsize=7)
fig.tight_layout()
U.save_fig(fig, "frequency_q19")
plt.show()


## LaTeX Table and Key Findings

In [ ]:
def render_latex(p: pd.DataFrame, label: str, n: int) -> list[str]:
    header = "\\multicolumn{4}{l}{\\textit{" + label + f" (n={n})" + "}} \\\\"
    out = [header]
    for _, r in p.iterrows():
        out.append(f"\\quad {r['label']} & {int(r['n'])} & {r['pct']:.0f}\\% & [{r['ci_lo']:.0f}--{r['ci_hi']:.0f}] \\\\")
    return out

lines = [
    "\\begin{table}[t]",
    "\\caption{Practices for incorporation, measurement, and documentation. Proportions with Bootstrap 95\\% CI.}",
    "\\label{tab:implementation-rq2}",
    "\\centering",
    "\\small",
    "\\begin{tabular}{lrrr}",
    "\\toprule",
    "\\textbf{Item} & \\textbf{n} & \\textbf{\\%} & \\textbf{IC 95\\%} \\\\",
    "\\midrule",
]
lines.extend(render_latex(p17, "Q17. Incorporation into development", n_q17))
lines.append("\\midrule")
lines.extend(render_latex(p18, "Q18. Measurement of model impact", n_q18))
lines.append("\\midrule")
lines.extend(render_latex(p20, "Q20. Documentation and communication", n_q20))
lines.extend(["\\bottomrule", "\\end{tabular}", "\\end{table}"])
(TABLES / "implementation_rq2.tex").write_text("\n".join(lines))
print("[saved] tables/implementation_rq2.tex")


In [ ]:
print("--- Q17 incorporation ---")
print(f"  Most common: {p17.iloc[0]['label']} ({p17.iloc[0]['pct']:.0f}%)")
print(f"  No formal strategy: {p17[p17['key']=='q17_no_strategy']['pct'].iloc[0]:.0f}%")

print("--- Q18 measurement ---")
print(f"  Most common: {p18.iloc[0]['label']} ({p18.iloc[0]['pct']:.0f}%)")
print(f"  A/B testing: {p18[p18['key']=='q18_ab_tests']['pct'].iloc[0]:.0f}%")

print("--- Q20 documentation ---")
print(f"  Most common: {p20.iloc[0]['label']} ({p20.iloc[0]['pct']:.0f}%)")
print(f"  Periodic reports: {p20[p20['key']=='q20_periodic_reports']['pct'].iloc[0]:.0f}%")
